# 05 — Model Evaluation

Detailed per-class analysis, confusion matrix, and feature importance.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

with open('../model/mindtype_model.pkl','rb') as f:
    art = pickle.load(f)
model = art['model']
le    = art['label_encoder']
FEAT_COLS = art['feature_names']

df = pd.read_csv('../data/processed/final_features.csv')
df_m = df[FEAT_COLS+['emotionIndex']].dropna()
X = df_m[FEAT_COLS].values
y = le.transform(df_m['emotionIndex'])
X_res, y_res = SMOTE(random_state=42, k_neighbors=3).fit_resample(X, y)
X_tr,X_te,y_tr,y_te = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

## Classification report

In [ ]:
print(classification_report(y_te, y_pred, target_names=le.classes_))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix — Random Forest')
plt.tight_layout()
plt.show()

## Feature importance

In [ ]:
fi = model.named_steps['clf'].feature_importances_
sorted_idx = np.argsort(fi)[::-1]
plt.figure(figsize=(7,3))
plt.bar([FEAT_COLS[i] for i in sorted_idx], [fi[i] for i in sorted_idx],
        color='#3266ad')
plt.title('Feature importance (Gini)')
plt.ylabel('Importance')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Binary stress detection

In [ ]:
y_stress = np.isin(le.inverse_transform(y_res), ['A','S']).astype(int)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
bin_model = Pipeline([('sc',StandardScaler()),('clf',RandomForestClassifier(200,class_weight='balanced',random_state=42))])
scores = cross_val_score(bin_model, X_res, y_stress, cv=StratifiedKFold(5,shuffle=True,random_state=42), scoring='f1')
print(f'Binary stress F1: {scores.mean():.4f} ± {scores.std():.4f}')